# MMCAD Training on Google Colab

**12 Multi-modal Retrieval Models** for CAD Sketch/Text → 3D Point Cloud

⚠️ **IMPORTANT**: This notebook requires **GPU runtime** and **Google Drive** for data storage

## Setup Checklist:
1. ✅ Enable GPU: Runtime → Change runtime type → GPU (T4 or better)
2. ✅ Mount Google Drive (Cell 2)
3. ✅ Upload dataset to Google Drive
4. ✅ Modify paths in Cell 3 (Configuration)

In [ ]:

# Cell 1: Install Packages (Fixed)
print("Installing packages...")

# Use --no-build-isolation to avoid building from source
!pip install -q --no-build-isolation transformers
!pip install -q plyfile tensorboard
!pip install -q opencv-python-headless

print("✓ Packages installed!")

# Check GPU
import torch
print(f"\n{'='*60}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"{'='*60}")



In [ ]:
# === IMPORTS ===
import os, sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
from transformers import ViTModel, BertTokenizer, BertModel, CLIPTokenizer, CLIPTextModel
from sklearn.model_selection import train_test_split
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import gc
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from plyfile import PlyData
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import gc
import time
import os
warnings.filterwarnings('ignore')

print("✓ Libraries imported!")


In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\n✓ Google Drive mounted at /content/drive/")
print("\nNOTE: Update paths in next cell (Configuration) to match your Google Drive structure")

In [ ]:
import os, time

print("="*60)
print("EXTRACTING SKETCHES")
print("="*60)

# MODIFY THIS PATH to match your Drive structure
SKETCHES_ZIP = "/content/drive/MyDrive/MMCAD/sketches.zip"
EXTRACT_DIR = "/content/mmcad_data"

os.makedirs(EXTRACT_DIR, exist_ok=True)

# Check if already extracted
sketch_dir = f"{EXTRACT_DIR}/abc_iso1_organized"
if os.path.exists(sketch_dir) and len(os.listdir(sketch_dir)) > 100:
    print(f"✓ Sketches already extracted ({len(os.listdir(sketch_dir))} files)")
else:
    print(f"Extracting from: {SKETCHES_ZIP}")
    start = time.time()
    !unzip -q "{SKETCHES_ZIP}" -d "{EXTRACT_DIR}"
    print(f"✓ Extracted in {time.time()-start:.1f}s")

print("="*60)

In [ ]:
import os, time

print("="*60)
print("EXTRACTING POINT CLOUDS")
print("="*60)

# MODIFY THIS PATH to match your Drive structure
PLY_TAR_LZ4 = "/content/drive/MyDrive/MMCAD/ply.tar.lz4"
EXTRACT_DIR = "/content/mmcad_data"

# Check if already extracted
ply_dir = f"{EXTRACT_DIR}/abc_ply_organized"
if os.path.exists(ply_dir) and len(os.listdir(ply_dir)) > 100:
    print(f"✓ Point clouds already extracted ({len(os.listdir(ply_dir))} files)")
else:
    print(f"Extracting from: {PLY_TAR_LZ4}")
    start = time.time()
    !apt-get install -qq lz4 > /dev/null 2>&1
    !lz4 -dc "{PLY_TAR_LZ4}" | tar -xf - -C "{EXTRACT_DIR}"
    print(f"✓ Extracted in {time.time()-start:.1f}s")

print("="*60)


In [ ]:
import os

print("="*60)
print("DATASET VERIFICATION")
print("="*60)

EXTRACT_DIR = "/content/mmcad_data"

# Check sketches
sketch_dir = f"{EXTRACT_DIR}/sketches"
if os.path.exists(sketch_dir):
    n_sketches = len(os.listdir(sketch_dir))
    print(f"✓ Sketches:      {n_sketches:6d} files")
else:
    print("✗ Sketches:      NOT FOUND")

# Check point clouds
ply_dir = f"{EXTRACT_DIR}/abc_ply_organized"
if os.path.exists(ply_dir):
    n_ply = len(os.listdir(ply_dir))
    print(f"✓ Point clouds:  {n_ply:6d} files")
else:
    print("✗ Point clouds:  NOT FOUND")

print("="*60)
print(f"\n✅ Dataset ready at: {EXTRACT_DIR}")
print(f"\n⚠️  IMPORTANT: Update Cell 3 Configuration:")
print(f'   Change DATA_ROOT to: "{EXTRACT_DIR}"')
print("="*60)


# MMCAD Training Notebook - Restructured

Train all 12 model variants on the MMCAD dataset with individual cells for each model:
- **Sketch-to-PointCloud** (ViT-based): 6 models
- **Text-to-PointCloud** (BERT-based): 3 models
- **Text-to-PointCloud** (CLIP-based): 3 models

**Each model has 2 cells**: Training cell → Validation cell → Memory cleanup

**Requirements**: `transformers`, `torch`, `torchvision`, `plyfile`, `sklearn`

In [ ]:
# Cell 3: Configuration Class (MODIFY PATHS FOR YOUR GOOGLE DRIVE)
import os

class Config:
    DATA_ROOT = "/content/mmcad_data"
    CSV_PATH = os.path.join(DATA_ROOT, "abc_dataset_clean.csv")
    OUTPUT_DIR = "/content/mmcad_training_checkpoints"
    SPLIT_DIR = "/content/mmcad_splits"

    TRAIN_RATIO = 0.80
    VAL_RATIO = 0.10
    TEST_RATIO = 0.10
    RANDOM_SEED = 42

    # A100 OPTIMIZED
    NUM_EPOCHS = 15
    BATCH_SIZE = 64        # ⬅️ 4x larger!
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    TEMPERATURE = 0.07
    LAMBDA_TRANSFORM = 0.001
    LR_STEP_SIZE = 10
    LR_GAMMA = 0.5

    NUM_POINTS = 2048
    LATENT_SIZE = 768
    LATENT_SIZE_CLIP = 512
    VIT_MODEL = 'google/vit-base-patch16-224'
    BERT_MODEL = 'bert-base-uncased'
    CLIP_MODEL = 'openai/clip-vit-base-patch32'
    DGCNN_K = 20

    SAVE_EVERY_N_EPOCHS = 10
    NUM_WORKERS = 2            # ⬅️ 2 is optimal
    K_VALUES = [1, 5, 10]

    SUBSET_SIZE = 141000
    ENABLE_TENSORBOARD = True

config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.SPLIT_DIR, exist_ok=True)


print(f"Configuration:")
print(f"{'='*60}")
print(f"Data root: {config.DATA_ROOT}")
print(f"Output directory: {config.OUTPUT_DIR}")
print(f"Split directory: {config.SPLIT_DIR}")
print(f"Data split: Train={config.TRAIN_RATIO*100:.0f}% / Val={config.VAL_RATIO*100:.0f}% / Test={config.TEST_RATIO*100:.0f}%")
print(f"Epochs: {config.NUM_EPOCHS}, Batch size: {config.BATCH_SIZE}")
print(f"{'='*60}")

In [ ]:
!cp /content/drive/MyDrive/MMCAD/abc_dataset_clean.csv /content/mmcad_data/

In [ ]:
# Cell 2: Configuration Class (Reconfigurable)
class Config:
    # ===========================================
    # DATA PATHS (Modify for your environment)
    # ===========================================
    # ⬇️ CHANGE THIS to match your Google Drive folder:
    DATA_ROOT = "/content/mmcad_data"

    # CSV file must be in DATA_ROOT
    CSV_PATH = os.path.join(DATA_ROOT, "abc_dataset_clean.csv")

    # Output directories (saved to Colab VM - fast but temporary)
    # NOTE: These will be deleted when session ends!
    # Copy to Drive after training (see instructions below)
    OUTPUT_DIR = "/content/mmcad_training_checkpoints"
    SPLIT_DIR = "/content/mmcad_splits"

    # ===========================================
    # DATA SPLIT CONFIGURATION
    # ===========================================
    TRAIN_RATIO = 0.80
    VAL_RATIO = 0.10
    TEST_RATIO = 0.10
    RANDOM_SEED = 42

    # ===========================================
    # TRAINING HYPERPARAMETERS
    # ===========================================
    NUM_EPOCHS = 15
    BATCH_SIZE = 64
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    TEMPERATURE = 0.07
    LAMBDA_TRANSFORM = 0.001
    LR_STEP_SIZE = 10
    LR_GAMMA = 0.5

    # ===========================================
    # MODEL CONFIGURATION
    # ===========================================
    NUM_POINTS = 2048
    LATENT_SIZE = 768
    LATENT_SIZE_CLIP = 512
    VIT_MODEL = 'google/vit-base-patch16-224'
    BERT_MODEL = 'bert-base-uncased'
    CLIP_MODEL = 'openai/clip-vit-base-patch32'
    DGCNN_K = 20

    # ===========================================
    # CHECKPOINT & LOGGING
    # ===========================================
    SAVE_EVERY_N_EPOCHS = 10
    NUM_WORKERS = 4
    K_VALUES = [1, 5, 10]

    # ===========================================
    # VALIDATION/DEBUG MODE
    # ===========================================
    SUBSET_SIZE = 141000 # Set to int for quick validation
    ENABLE_TENSORBOARD = True

config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.SPLIT_DIR, exist_ok=True)
print(f"Output directory: {config.OUTPUT_DIR}")
print(f"Split directory: {config.SPLIT_DIR}")
print(f"Data split: Train={config.TRAIN_RATIO*100:.0f}% / Val={config.VAL_RATIO*100:.0f}% / Test={config.TEST_RATIO*100:.0f}%")

In [ ]:
# Cell 3: PointNet Components
class STN3d(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(3, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 1024, 1)
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 9)
        self.bn1, self.bn2, self.bn3 = nn.BatchNorm1d(64), nn.BatchNorm1d(128), nn.BatchNorm1d(1024)
        self.bn4, self.bn5 = nn.BatchNorm1d(512), nn.BatchNorm1d(256)

    def forward(self, x):
        bs = x.size(0)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = torch.max(x, 2)[0]
        x = F.relu(self.bn4(self.fc1(x)))
        x = F.relu(self.bn5(self.fc2(x)))
        x = self.fc3(x)
        iden = torch.eye(3, device=x.device).flatten().unsqueeze(0).repeat(bs, 1)
        return (x + iden).view(-1, 3, 3)

class STNkd(nn.Module):
    def __init__(self, k=64):
        super().__init__()
        self.conv1 = nn.Conv1d(k, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 1024, 1)
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, k*k)
        self.bn1, self.bn2, self.bn3 = nn.BatchNorm1d(64), nn.BatchNorm1d(128), nn.BatchNorm1d(1024)
        self.bn4, self.bn5 = nn.BatchNorm1d(512), nn.BatchNorm1d(256)
        self.k = k

    def forward(self, x):
        bs = x.size(0)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = torch.max(x, 2)[0]
        x = F.relu(self.bn4(self.fc1(x)))
        x = F.relu(self.bn5(self.fc2(x)))
        x = self.fc3(x)
        iden = torch.eye(self.k, device=x.device).flatten().unsqueeze(0).repeat(bs, 1)
        return (x + iden).view(-1, self.k, self.k)

class PointNetEncoder(nn.Module):
    def __init__(self, output_dim=1024, feature_transform=True):
        super().__init__()
        self.stn = STN3d()
        self.conv1 = nn.Conv1d(3, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, output_dim, 1)
        self.bn1, self.bn2, self.bn3 = nn.BatchNorm1d(64), nn.BatchNorm1d(128), nn.BatchNorm1d(output_dim)
        self.dropout = nn.Dropout(0.3)
        self.output_dim = output_dim
        self.feature_transform = feature_transform
        if feature_transform:
            self.fstn = STNkd(k=64)

    def forward(self, x):
        trans = self.stn(x)
        x = torch.bmm(x.transpose(2, 1), trans).transpose(2, 1)
        x = F.relu(self.bn1(self.conv1(x)))
        trans_feat = None
        if self.feature_transform:
            trans_feat = self.fstn(x)
            x = torch.bmm(x.transpose(2, 1), trans_feat).transpose(2, 1)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.bn3(self.conv3(x))
        x = torch.max(x, 2)[0]
        return self.dropout(x), trans, trans_feat

In [ ]:
# Cell 4: PointNet-Mini
class PointCloudEncoder(nn.Module):
    def __init__(self, output_dim=1024):
        super().__init__()
        self.mlp1 = nn.Conv1d(3, 64, 1)
        self.mlp2 = nn.Conv1d(64, 128, 1)
        self.mlp3 = nn.Conv1d(128, 256, 1)
        self.mlp4 = nn.Conv1d(256, output_dim, 1)
        self.bn1, self.bn2 = nn.BatchNorm1d(64), nn.BatchNorm1d(128)
        self.bn3, self.bn4 = nn.BatchNorm1d(256), nn.BatchNorm1d(output_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = F.relu(self.bn1(self.mlp1(x)))
        x = F.relu(self.bn2(self.mlp2(x)))
        x = F.relu(self.bn3(self.mlp3(x)))
        x = F.relu(self.bn4(self.mlp4(x)))
        return self.dropout(torch.max(x, 2)[0]), None, None

In [ ]:
# Cell 5: DGCNN Encoder
def knn(x, k):
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x**2, dim=1, keepdim=True)
    return (-xx - inner - xx.transpose(2, 1)).topk(k=k, dim=-1)[1]

def get_graph_feature(x, k=20):
    bs, d, n = x.size()
    idx = knn(x, k)
    idx_base = torch.arange(0, bs, device=x.device).view(-1, 1, 1) * n
    idx = (idx + idx_base).view(-1)
    x = x.transpose(2, 1).contiguous()
    feat = x.view(bs * n, -1)[idx].view(bs, n, k, d)
    x = x.view(bs, n, 1, d).repeat(1, 1, k, 1)
    return torch.cat((feat - x, x), dim=3).permute(0, 3, 1, 2).contiguous()

class DGCNNEncoder(nn.Module):
    def __init__(self, latent_size=1024, k=20):
        super().__init__()
        self.k = k
        self.bn1, self.bn2 = nn.BatchNorm2d(64), nn.BatchNorm2d(64)
        self.bn3, self.bn4 = nn.BatchNorm2d(128), nn.BatchNorm2d(256)
        self.bn5, self.bn6, self.bn7 = nn.BatchNorm1d(latent_size), nn.BatchNorm1d(512), nn.BatchNorm1d(latent_size)
        self.conv1 = nn.Sequential(nn.Conv2d(6, 64, 1, bias=False), self.bn1, nn.LeakyReLU(0.2))
        self.conv2 = nn.Sequential(nn.Conv2d(128, 64, 1, bias=False), self.bn2, nn.LeakyReLU(0.2))
        self.conv3 = nn.Sequential(nn.Conv2d(128, 128, 1, bias=False), self.bn3, nn.LeakyReLU(0.2))
        self.conv4 = nn.Sequential(nn.Conv2d(256, 256, 1, bias=False), self.bn4, nn.LeakyReLU(0.2))
        self.conv5 = nn.Sequential(nn.Conv1d(512, latent_size, 1, bias=False), self.bn5, nn.LeakyReLU(0.2))
        self.linear1 = nn.Linear(latent_size * 2, 512, bias=False)
        self.linear2 = nn.Linear(512, latent_size)
        self.dp = nn.Dropout(0.3)

    def forward(self, x):
        bs = x.size(0)
        x1 = self.conv1(get_graph_feature(x, self.k)).max(dim=-1)[0]
        x2 = self.conv2(get_graph_feature(x1, self.k)).max(dim=-1)[0]
        x3 = self.conv3(get_graph_feature(x2, self.k)).max(dim=-1)[0]
        x4 = self.conv4(get_graph_feature(x3, self.k)).max(dim=-1)[0]
        x = self.conv5(torch.cat((x1, x2, x3, x4), dim=1))
        x = torch.cat((x.max(2)[0], x.mean(2)), 1)
        x = self.dp(F.leaky_relu(self.bn6(self.linear1(x)), 0.2))
        return self.dp(self.bn7(self.linear2(x))), None, None

In [ ]:
# Cell 6: ViT Sketch Encoder
class ViTSketchEncoder(nn.Module):
    def __init__(self, model_name='google/vit-base-patch16-224', use_projection=True, latent_size=768):
        super().__init__()
        self.vit = ViTModel.from_pretrained(model_name)
        self.use_projection = use_projection
        if use_projection:
            self.projection = nn.Linear(self.vit.config.hidden_size, latent_size)
            self.output_dim = latent_size
        else:
            self.output_dim = self.vit.config.hidden_size

    def forward(self, x):
        x = self.vit(pixel_values=x).last_hidden_state[:, 0]
        return self.projection(x) if self.use_projection else x

In [ ]:
# Cell 7: SketchPointCloudNetwork
class SketchPointCloudNetwork(nn.Module):
    def __init__(self, pc_encoder_type='pointnet', latent_size=768,
                 use_sketch_projection=True, use_pc_projection=True, freeze_vit=False,
                 feature_transform=True, dgcnn_k=20):
        super().__init__()
        self.sketch_encoder = ViTSketchEncoder(use_projection=use_sketch_projection, latent_size=latent_size)
        if pc_encoder_type == 'dgcnn':
            self.pc_encoder = DGCNNEncoder(latent_size=1024, k=dgcnn_k)
        elif pc_encoder_type == 'pointcloud':
            self.pc_encoder = PointCloudEncoder(output_dim=1024)
        else:
            self.pc_encoder = PointNetEncoder(output_dim=1024, feature_transform=feature_transform)
        self.use_pc_projection = use_pc_projection
        if use_pc_projection:
            self.pointcloud_projection = nn.Linear(1024, latent_size)
        if freeze_vit:
            for p in self.sketch_encoder.vit.parameters():
                p.requires_grad = False

    def forward(self, sketch, pc):
        sketch_feat = self.sketch_encoder(sketch)
        pc_feat, trans, trans_feat = self.pc_encoder(pc)
        if self.use_pc_projection:
            pc_feat = self.pointcloud_projection(pc_feat)
        return {'sketch_features': sketch_feat, 'pc_features': pc_feat, 'trans': trans, 'trans_feat': trans_feat}

In [ ]:
# Cell 8: Text Encoders & TextPointCloudNetwork
class BertTextEncoder(nn.Module):
    def __init__(self, model_name='bert-base-uncased', latent_size=768):
        super().__init__()
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.bert = BertModel.from_pretrained(model_name)
        self.projection = nn.Linear(768, latent_size)

    def forward(self, text_tokens):
        outputs = self.bert(**text_tokens)
        return self.projection(outputs.last_hidden_state[:, 0, :])

class CLIPTextEncoder(nn.Module):
    def __init__(self, model_name='openai/clip-vit-base-patch32', latent_size=512):
        super().__init__()
        self.tokenizer = CLIPTokenizer.from_pretrained(model_name)
        self.text_model = CLIPTextModel.from_pretrained(model_name)
        self.projection = nn.Linear(self.text_model.config.hidden_size, latent_size)
        self.bn = nn.BatchNorm1d(latent_size)
        self.dropout = nn.Dropout(0.3)

    def forward(self, text_tokens):
        x = self.text_model(**text_tokens).pooler_output
        return self.dropout(self.bn(self.projection(x)))

class TextPointCloudNetwork(nn.Module):
    def __init__(self, text_encoder_type='bert', pc_encoder_type='dgcnn', latent_size=768,
                 freeze_text=False, feature_transform=True, dgcnn_k=20):
        super().__init__()
        if text_encoder_type == 'clip':
            self.text_encoder = CLIPTextEncoder(latent_size=latent_size)
        else:
            self.text_encoder = BertTextEncoder(latent_size=latent_size)
        if pc_encoder_type == 'dgcnn':
            self.pc_encoder = DGCNNEncoder(latent_size=1024, k=dgcnn_k)
        elif pc_encoder_type == 'pointcloud':
            self.pc_encoder = PointCloudEncoder(output_dim=1024)
        else:
            self.pc_encoder = PointNetEncoder(output_dim=1024, feature_transform=feature_transform)
        self.pc_projection = nn.Linear(1024, latent_size)
        if freeze_text:
            encoder = self.text_encoder.bert if text_encoder_type == 'bert' else self.text_encoder.text_model
            for p in encoder.parameters():
                p.requires_grad = False

    def forward(self, text_tokens, pc):
        text_feat = self.text_encoder(text_tokens)
        pc_feat, trans, trans_feat = self.pc_encoder(pc)
        return {'text_features': text_feat, 'pc_features': self.pc_projection(pc_feat), 'trans': trans, 'trans_feat': trans_feat}

In [ ]:
# Cell 9: Loss Functions
def info_nce_loss(features_a, features_b, temperature=0.07):
    features_a = F.normalize(features_a, dim=1)
    features_b = F.normalize(features_b, dim=1)
    labels = torch.arange(len(features_a), device=features_a.device)
    logits_a2b = torch.mm(features_a, features_b.t()) / temperature
    logits_b2a = torch.mm(features_b, features_a.t()) / temperature
    return (F.cross_entropy(logits_a2b, labels) + F.cross_entropy(logits_b2a, labels)) / 2.0

def feature_transform_regularizer(trans):
    if trans is None:
        return 0.0
    d = trans.size()[1]
    I = torch.eye(d, device=trans.device).unsqueeze(0)
    return torch.mean(torch.norm(torch.bmm(trans, trans.transpose(2, 1)) - I, dim=(1, 2)))

In [ ]:
# Cell 10: Metrics Calculation
def calc_metrics(sim_matrix, k_values=[1, 5, 10]):
    sim = sim_matrix.cpu().numpy() if isinstance(sim_matrix, torch.Tensor) else sim_matrix
    n = sim.shape[0]
    sorted_idx = np.argsort(-sim, axis=1)
    metrics = {}
    for k in k_values:
        recalls, aps = [], []
        for i in range(n):
            top_k = sorted_idx[i, :k]
            hit = int(i in top_k)
            recalls.append(hit)
            aps.append(1.0 / (np.where(top_k == i)[0][0] + 1) if hit else 0.0)
        metrics[f'recall@{k}'] = np.mean(recalls) * 100
        metrics[f'mAP@{k}'] = np.mean(aps) * 100
    return metrics

@torch.no_grad()
def evaluate_sketch_model(model, loader, device, k_values=[1, 5, 10]):
    model.eval()
    s_feats, p_feats = [], []
    for batch in tqdm(loader, desc="Evaluating", leave=False):
        out = model(batch['sketch'].to(device), batch['point_cloud'].to(device))
        s_feats.append(out['sketch_features'].cpu())
        p_feats.append(out['pc_features'].cpu())
    s = F.normalize(torch.cat(s_feats), dim=1)
    p = F.normalize(torch.cat(p_feats), dim=1)
    return calc_metrics(s @ p.T, k_values)

@torch.no_grad()
def evaluate_text_model(model, loader, device, k_values=[1, 5, 10]):
    model.eval()
    t_feats, p_feats = [], []
    for batch in tqdm(loader, desc="Evaluating", leave=False):
        text_tokens = {k: v.to(device) for k, v in batch['text_tokens'].items()}
        out = model(text_tokens, batch['point_cloud'].to(device))
        t_feats.append(out['text_features'].cpu())
        p_feats.append(out['pc_features'].cpu())
    t = F.normalize(torch.cat(t_feats), dim=1)
    p = F.normalize(torch.cat(p_feats), dim=1)
    return calc_metrics(t @ p.T, k_values)

In [ ]:
def load_ply_fast(path):
    """Fast PLY loading"""
    plydata = PlyData.read(path)
    return np.stack([plydata['vertex']['x'],
                    plydata['vertex']['y'],
                    plydata['vertex']['z']], axis=1).astype(np.float32)


class OptimizedSketchDataset(Dataset):
    """
    Optimized dataset:
    - Uses OpenCV (faster than PIL)
    - Caches PLY data in RAM (17GB)
    - Simplified transforms
    """

    def __init__(self, df, config, cache_ply=True):
        self.config = config
        self.data_root = config.DATA_ROOT

        # Simplified transform (no Grayscale conversion)
        self.normalize_mean = np.array([0.5, 0.5, 0.5], dtype=np.float32)
        self.normalize_std = np.array([0.5, 0.5, 0.5], dtype=np.float32)

        # Validation
        print(f"Validating {len(df)} samples...")
        valid_indices = []
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Validating"):
            iso1 = os.path.join(self.data_root, row['sketch_iso1_path'])
            iso2 = os.path.join(self.data_root, row['sketch_iso2_path'])
            ply = os.path.join(self.data_root, row['ply_path'])
            if (os.path.isfile(iso1) or os.path.isfile(iso2)) and os.path.isfile(ply):
                valid_indices.append(idx)

        self.samples = df.loc[valid_indices].reset_index(drop=True)
        print(f"✓ Valid: {len(self.samples)}/{len(df)} ({len(self.samples)/len(df)*100:.1f}%)")

        # Cache PLY data (fits in ~17GB)
        self.cache_ply = cache_ply
        if cache_ply:
            print(f"\nCaching PLY data (~17GB)...")
            self.ply_cache = {}
            for idx in tqdm(range(len(self.samples)), desc="Caching PLY"):
                row = self.samples.iloc[idx]
                try:
                    ply_path = os.path.join(self.data_root, row['ply_path'])
                    xyz = load_ply_fast(ply_path)
                    # Pre-normalize
                    xyz = xyz - xyz.mean(0)
                    norm = np.max(np.linalg.norm(xyz, axis=1))
                    if norm > 0:
                        xyz = xyz / norm
                    self.ply_cache[idx] = xyz
                except:
                    self.ply_cache[idx] = np.zeros((10000, 3), dtype=np.float32)

            mem_gb = sum(p.nbytes for p in self.ply_cache.values()) / 1e9
            print(f"✓ Cached {len(self.ply_cache)} PLY files ({mem_gb:.1f} GB)")

        gc.collect()

    def __len__(self):
        return len(self.samples)

    def _load_image_fast(self, path):
        """Fast image loading with OpenCV"""
        img = cv2.imread(path)
        if img is None:
            return None
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        # Convert to tensor and normalize
        img = img.astype(np.float32) / 255.0
        img = (img - self.normalize_mean) / self.normalize_std
        return torch.from_numpy(img).permute(2, 0, 1)

    def __getitem__(self, i):
        row = self.samples.iloc[i]

        # Load image with OpenCV (faster than PIL)
        sketch = None
        for col in ['sketch_iso1_path', 'sketch_iso2_path']:
            try:
                path = os.path.join(self.data_root, row[col])
                sketch = self._load_image_fast(path)
                if sketch is not None:
                    break
            except:
                continue

        if sketch is None:
            sketch = torch.zeros(3, 224, 224)

        # Get PLY (from cache or disk)
        if self.cache_ply:
            pc = self.ply_cache[i]
        else:
            ply_path = os.path.join(self.data_root, row['ply_path'])
            pc = load_ply_fast(ply_path)
            pc = pc - pc.mean(0)
            norm = np.max(np.linalg.norm(pc, axis=1))
            if norm > 0:
                pc = pc / norm

        # Random sampling
        idx = np.random.choice(len(pc), self.config.NUM_POINTS,
                              replace=len(pc) < self.config.NUM_POINTS)
        pc = torch.from_numpy(pc[idx]).T

        return {'sketch': sketch, 'point_cloud': pc, 'idx': i}


class OptimizedTextDataset(Dataset):
    """Optimized text dataset with PLY caching"""

    def __init__(self, df, config, tokenizer, cache_ply=True):
        self.config = config
        self.data_root = config.DATA_ROOT
        self.samples = df.reset_index(drop=True)
        self.tokenizer = tokenizer

        # Cache PLY
        self.cache_ply = cache_ply
        if cache_ply:
            print(f"Caching PLY data for text dataset...")
            self.ply_cache = {}
            for idx in tqdm(range(len(self.samples)), desc="Caching PLY"):
                row = self.samples.iloc[idx]
                try:
                    ply_path = os.path.join(self.data_root, row['ply_path'])
                    xyz = load_ply_fast(ply_path)
                    xyz = xyz - xyz.mean(0)
                    norm = np.max(np.linalg.norm(xyz, axis=1))
                    if norm > 0:
                        xyz = xyz / norm
                    self.ply_cache[idx] = xyz
                except:
                    self.ply_cache[idx] = np.zeros((10000, 3), dtype=np.float32)

        print(f"✓ TextDataset ready: {len(self.samples)} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        row = self.samples.iloc[i]

        # Get PLY
        if self.cache_ply:
            pc = self.ply_cache[i]
        else:
            ply_path = os.path.join(self.data_root, row['ply_path'])
            pc = load_ply_fast(ply_path)
            pc = pc - pc.mean(0)
            norm = np.max(np.linalg.norm(pc, axis=1))
            if norm > 0:
                pc = pc / norm

        idx = np.random.choice(len(pc), self.config.NUM_POINTS,
                              replace=len(pc) < self.config.NUM_POINTS)
        pc = torch.from_numpy(pc[idx]).T

        # Tokenize
        text = f"{row['title']}. {row['description']}"
        text_tokens = self.tokenizer(text, max_length=128, padding='max_length',
                                    truncation=True, return_tensors='pt')
        text_tokens = {k: v.squeeze(0) for k, v in text_tokens.items()}

        return {'text_tokens': text_tokens, 'point_cloud': pc, 'idx': i}

In [ ]:
def create_dataloaders(train_dataset, val_dataset, config):
    """Create optimized dataloaders"""

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.BATCH_SIZE,  # 16
        shuffle=True,
        num_workers=config.NUM_WORKERS,  # 8
        pin_memory=True,
        drop_last=True
        # NO persistent_workers (causes crashes)
        # NO prefetch_factor (causes crashes)
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=4,  # Lower for validation
        pin_memory=True
    )

    print(f"DataLoaders created:")
    print(f"  Train: {len(train_dataset)} samples, {len(train_loader)} batches")
    print(f"  Val:   {len(val_dataset)} samples, {len(val_loader)} batches")
    print(f"  Batch size: {config.BATCH_SIZE}, Workers: {config.NUM_WORKERS}")

    return train_loader, val_loader

In [ ]:
# Cell 12: Data Loading with Train/Val/Test Split
df = pd.read_csv(config.CSV_PATH)
print(f"Total entries in CSV: {len(df)}")

# Filter for COMPLETE entries only (all files exist)
valid_df = df[df['completeness_score'] == 7].copy()
print(f"Complete entries (score=7): {len(valid_df)}")

# Additional check (optional, for safety)
valid_df = valid_df[
    (valid_df['has_iso1'] == True) &
    (valid_df['has_iso2'] == True) &
    (valid_df['has_ply'] == True)
].copy()
print(f"Valid entries after verification: {len(valid_df)}")

if config.SUBSET_SIZE:
    valid_df = valid_df.sample(config.SUBSET_SIZE, random_state=config.RANDOM_SEED)
    print(f"Using subset of {config.SUBSET_SIZE} samples")

train_split_path = os.path.join(config.SPLIT_DIR, 'train_uids.txt')
val_split_path = os.path.join(config.SPLIT_DIR, 'val_uids.txt')
test_split_path = os.path.join(config.SPLIT_DIR, 'test_uids.txt')

if os.path.exists(train_split_path) and os.path.exists(val_split_path) and os.path.exists(test_split_path):
    print("Loading existing split files...")
    with open(train_split_path) as f: train_uids = set(int(line.strip()) for line in f)
    with open(val_split_path) as f: val_uids = set(int(line.strip()) for line in f)
    with open(test_split_path) as f: test_uids = set(int(line.strip()) for line in f)
    train_df = valid_df[valid_df['uid'].isin(train_uids)]
    val_df = valid_df[valid_df['uid'].isin(val_uids)]
    test_df = valid_df[valid_df['uid'].isin(test_uids)]
else:
    print("Creating new train/val/test splits...")
    train_df, temp_df = train_test_split(valid_df, train_size=config.TRAIN_RATIO, random_state=config.RANDOM_SEED)
    val_ratio_adjusted = config.VAL_RATIO / (config.VAL_RATIO + config.TEST_RATIO)
    val_df, test_df = train_test_split(temp_df, train_size=val_ratio_adjusted, random_state=config.RANDOM_SEED)
    with open(train_split_path, 'w') as f:
        for uid in train_df['uid']: f.write(f"{uid}\n")
    with open(val_split_path, 'w') as f:
        for uid in val_df['uid']: f.write(f"{uid}\n")
    with open(test_split_path, 'w') as f:
        for uid in test_df['uid']: f.write(f"{uid}\n")
    print(f"Split files saved to {config.SPLIT_DIR}")

print(f"\nData Split:")
print(f"  Train: {len(train_df)} samples ({len(train_df)/len(valid_df)*100:.1f}%)")
print(f"  Val:   {len(val_df)} samples ({len(val_df)/len(valid_df)*100:.1f}%)")
print(f"  Test:  {len(test_df)} samples ({len(test_df)/len(valid_df)*100:.1f}%)")


In [ ]:
# Cell 13: Training & Checkpoint Utilities (FIXED)

def train_epoch_sketch(model, loader, optimizer, device, temperature, lambda_transform):
    model.train()
    total_loss = 0
    pbar = tqdm(loader, desc='Training', leave=False)
    for i, batch in enumerate(pbar):  # ← FIXED: Added enumerate
        optimizer.zero_grad()
        outputs = model(batch['sketch'].to(device), batch['point_cloud'].to(device))
        loss = info_nce_loss(outputs['sketch_features'], outputs['pc_features'], temperature)
        if lambda_transform > 0 and outputs.get('trans_feat') is not None:
            loss += lambda_transform * feature_transform_regularizer(outputs['trans_feat'])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if i % 20 == 0:
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    return total_loss / len(loader)


def train_epoch_text(model, loader, optimizer, device, temperature, lambda_transform):
    model.train()
    total_loss = 0
    pbar = tqdm(loader, desc='Training', leave=False)
    for i, batch in enumerate(pbar):  # ← FIXED: Added enumerate
        text_tokens = {k: v.to(device) for k, v in batch['text_tokens'].items()}
        optimizer.zero_grad()
        outputs = model(text_tokens, batch['point_cloud'].to(device))
        loss = info_nce_loss(outputs['text_features'], outputs['pc_features'], temperature)
        if lambda_transform > 0 and outputs.get('trans_feat') is not None:
            loss += lambda_transform * feature_transform_regularizer(outputs['trans_feat'])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if i % 20 == 0:
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    return total_loss / len(loader)


@torch.no_grad()
def validate_epoch_sketch(model, loader, device, temperature):
    model.eval()
    total_loss = 0
    for batch in loader:
        out = model(batch['sketch'].to(device), batch['point_cloud'].to(device))
        total_loss += info_nce_loss(out['sketch_features'], out['pc_features'], temperature).item()
    return total_loss / len(loader)


@torch.no_grad()
def validate_epoch_text(model, loader, device, temperature):
    model.eval()
    total_loss = 0
    for batch in loader:
        text_tokens = {k: v.to(device) for k, v in batch['text_tokens'].items()}
        out = model(text_tokens, batch['point_cloud'].to(device))
        total_loss += info_nce_loss(out['text_features'], out['pc_features'], temperature).item()
    return total_loss / len(loader)


def save_checkpoint(model, optimizer, scheduler, epoch, metrics, save_dir, filename='checkpoint.pth'):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'metrics': metrics
    }, os.path.join(save_dir, filename))


def load_checkpoint(model, optimizer, scheduler, checkpoint_path):
    if not os.path.exists(checkpoint_path):
        return 0
    # FIXED: Added weights_only=False for PyTorch 2.6+
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    state_dict = {k.replace('module.', ''): v for k, v in checkpoint.get('model_state_dict', checkpoint).items()}
    model.load_state_dict(state_dict, strict=False)
    if optimizer and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if scheduler and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    return checkpoint.get('epoch', 0) + 1


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {'total': total, 'trainable': trainable}


def plot_training_curves(metrics_df, save_dir):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(metrics_df['epoch'], metrics_df['train_loss'], label='Train')
    axes[0].plot(metrics_df['epoch'], metrics_df['val_loss'], label='Val')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Loss'); axes[0].legend()
    for k in [1, 5, 10]:
        if f'val_recall@{k}' in metrics_df.columns:
            axes[1].plot(metrics_df['epoch'], metrics_df[f'val_recall@{k}'], label=f'R@{k}')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Recall'); axes[1].set_title('Recall'); axes[1].legend()
    for k in [1, 5, 10]:
        if f'val_mAP@{k}' in metrics_df.columns:
            axes[2].plot(metrics_df['epoch'], metrics_df[f'val_mAP@{k}'], label=f'mAP@{k}')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('mAP'); axes[2].set_title('mAP'); axes[2].legend()
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'training_curves.png'), dpi=150)
    plt.close()


def cleanup_memory():
    """Aggressive memory cleanup after training"""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    print(f"Memory cleaned. GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Cell 3.5: Device Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")



---
## 💾 IMPORTANT: Save Checkpoints to Google Drive

**Colab sessions disconnect after 12 hours**. To prevent data loss:

```python
# After training each model, copy checkpoints to Google Drive:
!cp -r /content/mmcad_training_checkpoints/* /content/drive/MyDrive/mmcad_checkpoints/
```

Or modify `config.OUTPUT_DIR` in Cell 3 to save directly to Drive:
```python
OUTPUT_DIR = "/content/drive/MyDrive/mmcad_checkpoints"
```

⚠️ **Warning**: Saving directly to Drive is slower but safer for long training runs.

---\n# Sketch-to-PointCloud Models\n## 6 models using ViT for sketch encoding

In [ ]:
# Create datasets
train_dataset = OptimizedSketchDataset(train_df, config, cache_ply=True)
val_dataset = OptimizedSketchDataset(val_df, config, cache_ply=True)
train_loader, val_loader = create_dataloaders(train_dataset, val_dataset, config)

In [ ]:
# Cell 15: Train Model 1 - ViT-PointNetMini (proj)
model_name = 'ViT-PointNetMini (proj)'
print(f"\\n{'='*80}\\nTraining: {model_name}\\n{'='*80}")

save_dir = os.path.join(config.OUTPUT_DIR, model_name.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))
os.makedirs(save_dir, exist_ok=True)

'''# Create datasets
train_dataset = OptimizedSketchDataset(train_df, config, cache_ply=True)
val_dataset = OptimizedSketchDataset(val_df, config, cache_ply=True)
train_loader, val_loader = create_dataloaders(train_dataset, val_dataset, config)'''

# Create model
model_1 = SketchPointCloudNetwork(
    pc_encoder_type='pointcloud',
    latent_size=config.LATENT_SIZE,
    use_sketch_projection=True,
    use_pc_projection=True,
    freeze_vit=False,
    dgcnn_k=config.DGCNN_K
).to(device)

params = count_parameters(model_1)
print(f"Parameters: Total={params['total']:,}, Trainable={params['trainable']:,}")

# Setup training
optimizer = optim.Adam(model_1.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=config.LR_STEP_SIZE, gamma=config.LR_GAMMA)
writer = SummaryWriter(log_dir=os.path.join(save_dir, 'logs')) if config.ENABLE_TENSORBOARD else None

metrics_history = []
best_recall = 0.0
lambda_transform = 0.0

# Training loop
for epoch in range(config.NUM_EPOCHS):
    print(f"\\nEpoch {epoch+1}/{config.NUM_EPOCHS}")
    train_loss = train_epoch_sketch(model_1, train_loader, optimizer, device, config.TEMPERATURE, lambda_transform)
    val_loss = validate_epoch_sketch(model_1, val_loader, device, config.TEMPERATURE)
    retrieval_metrics = evaluate_sketch_model(model_1, val_loader, device, config.K_VALUES)
    scheduler.step()

    print(f"  Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    print(f"  Val Recall@1: {retrieval_metrics['recall@1']:.2f}, mAP@1: {retrieval_metrics['mAP@1']:.2f}")

    epoch_metrics = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                     **{f'val_{k}': v for k, v in retrieval_metrics.items()}}
    metrics_history.append(epoch_metrics)

    if writer:
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        for k, v in retrieval_metrics.items():
            writer.add_scalar(f'Val/{k}', v, epoch)

    if retrieval_metrics['recall@1'] > best_recall:
        best_recall = retrieval_metrics['recall@1']
        save_checkpoint(model_1, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'best.pth')
        print(f"  New best Recall@1: {best_recall:.2f}")

    save_checkpoint(model_1, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'latest.pth')
    if (epoch + 1) % config.SAVE_EVERY_N_EPOCHS == 0:
        save_checkpoint(model_1, optimizer, scheduler, epoch, retrieval_metrics, save_dir, f'epoch_{epoch+1}.pth')

# Save metrics
metrics_df = pd.DataFrame(metrics_history)
metrics_df.to_csv(os.path.join(save_dir, 'metrics.csv'), index=False)
plot_training_curves(metrics_df, save_dir)

if writer: writer.close()

print(f"\\n{'='*80}\\n{model_name} training complete!\\nBest Val Recall@1: {best_recall:.2f}%\\n{'='*80}")

In [ ]:
# Cell 16: Validate Model 1 - ViT-PointNetMini (proj)
print(f"\\nFinal validation for: {model_name}")
print(f"{'='*80}")

# Load best checkpoint
load_checkpoint(model_1, None, None, os.path.join(save_dir, 'best.pth'))

# Evaluate on validation set
final_metrics = evaluate_sketch_model(model_1, val_loader, device, config.K_VALUES)

print(f"\\nFinal Validation Results:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.2f}%")


In [ ]:
all_training_results=[]
# Store results
all_training_results.append({
    'name': model_name,
    'type': 'Sketch',
    'best_val_recall@1': best_recall,
    'final_metrics': final_metrics,
    'save_dir': save_dir
})

# Cleanup
del model_1, optimizer, scheduler, train_loader, val_loader, train_dataset, val_dataset
if writer: del writer
cleanup_memory()

print(f"\\n{'='*80}\\nModel 1 complete and memory cleaned!\\n{'='*80}")

In [ ]:
cleanup_memory()

In [ ]:
# Cell 17: Train Model 2 - ViT-PointNetMini (no proj)
model_name = 'ViT-PointNetMini (no proj)'
print(f"\\n{'='*80}\\nTraining: {model_name}\\n{'='*80}")

save_dir = os.path.join(config.OUTPUT_DIR, model_name.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))
os.makedirs(save_dir, exist_ok=True)

'''# Create datasets
train_dataset = MMCADSketchDataset(train_df, config)
val_dataset = MMCADSketchDataset(val_df, config)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
                          num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                        num_workers=config.NUM_WORKERS, pin_memory=True)'''


# Create model
model_2 = SketchPointCloudNetwork(
    pc_encoder_type='pointcloud',
    latent_size=config.LATENT_SIZE,
    use_sketch_projection=False,
    use_pc_projection=True,
    freeze_vit=False,
    dgcnn_k=config.DGCNN_K
).to(device)

params = count_parameters(model_2)
print(f"Parameters: Total={params['total']:,}, Trainable={params['trainable']:,}")

# Setup training
optimizer = optim.Adam(model_2.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=config.LR_STEP_SIZE, gamma=config.LR_GAMMA)
writer = SummaryWriter(log_dir=os.path.join(save_dir, 'logs')) if config.ENABLE_TENSORBOARD else None

metrics_history = []
best_recall = 0.0
lambda_transform = 0.0

# Training loop
for epoch in range(config.NUM_EPOCHS):
    print(f"\\nEpoch {epoch+1}/{config.NUM_EPOCHS}")
    train_loss = train_epoch_sketch(model_2, train_loader, optimizer, device, config.TEMPERATURE, lambda_transform)
    val_loss = validate_epoch_sketch(model_2, val_loader, device, config.TEMPERATURE)
    retrieval_metrics = evaluate_sketch_model(model_2, val_loader, device, config.K_VALUES)
    scheduler.step()

    print(f"  Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    print(f"  Val Recall@1: {retrieval_metrics['recall@1']:.2f}, mAP@1: {retrieval_metrics['mAP@1']:.2f}")

    epoch_metrics = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                     **{f'val_{k}': v for k, v in retrieval_metrics.items()}}
    metrics_history.append(epoch_metrics)

    if writer:
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        for k, v in retrieval_metrics.items():
            writer.add_scalar(f'Val/{k}', v, epoch)

    if retrieval_metrics['recall@1'] > best_recall:
        best_recall = retrieval_metrics['recall@1']
        save_checkpoint(model_2, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'best.pth')
        print(f"  New best Recall@1: {best_recall:.2f}")

    save_checkpoint(model_2, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'latest.pth')
    if (epoch + 1) % config.SAVE_EVERY_N_EPOCHS == 0:
        save_checkpoint(model_2, optimizer, scheduler, epoch, retrieval_metrics, save_dir, f'epoch_{epoch+1}.pth')

# Save metrics
metrics_df = pd.DataFrame(metrics_history)
metrics_df.to_csv(os.path.join(save_dir, 'metrics.csv'), index=False)
plot_training_curves(metrics_df, save_dir)

if writer: writer.close()

print(f"\\n{'='*80}\\n{model_name} training complete!\\nBest Val Recall@1: {best_recall:.2f}%\\n{'='*80}")

In [ ]:
cleanup_memory()

In [ ]:
# Cell 18: Validate Model 2 - ViT-PointNetMini (no proj)
print(f"\\nFinal validation for: {model_name}")
print(f"{'='*80}")

# Load best checkpoint
load_checkpoint(model_2, None, None, os.path.join(save_dir, 'best.pth'))

# Evaluate on validation set
final_metrics = evaluate_sketch_model(model_2, val_loader, device, config.K_VALUES)

print(f"\\nFinal Validation Results:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.2f}%")

# Store results
all_training_results.append({
    'name': model_name,
    'type': 'Sketch',
    'best_val_recall@1': best_recall,
    'final_metrics': final_metrics,
    'save_dir': save_dir
})

# Cleanup
del model_2, optimizer, scheduler, train_loader, val_loader, train_dataset, val_dataset
if writer: del writer
cleanup_memory()

print(f"\\n{'='*80}\\nModel 2 complete and memory cleaned!\\n{'='*80}")

In [ ]:
# Cell 19: Train Model 3 - ViT-PointNet (proj)
model_name = 'ViT-PointNet (proj)'
print(f"\\n{'='*80}\\nTraining: {model_name}\\n{'='*80}")

save_dir = os.path.join(config.OUTPUT_DIR, model_name.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))
os.makedirs(save_dir, exist_ok=True)

'''# Create datasets
train_dataset = MMCADSketchDataset(train_df, config)
val_dataset = MMCADSketchDataset(val_df, config)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
                          num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                        num_workers=config.NUM_WORKERS, pin_memory=True)'''
train_dataset = OptimizedSketchDataset(train_df, config, cache_ply=True)
val_dataset = OptimizedSketchDataset(val_df, config, cache_ply=True)
train_loader, val_loader = create_dataloaders(train_dataset, val_dataset, config)

# Create model
model_3 = SketchPointCloudNetwork(
    pc_encoder_type='pointnet',
    latent_size=config.LATENT_SIZE,
    use_sketch_projection=True,
    use_pc_projection=True,
    freeze_vit=False,
    dgcnn_k=config.DGCNN_K
).to(device)

params = count_parameters(model_3)
print(f"Parameters: Total={params['total']:,}, Trainable={params['trainable']:,}")

# Setup training
optimizer = optim.Adam(model_3.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=config.LR_STEP_SIZE, gamma=config.LR_GAMMA)
writer = SummaryWriter(log_dir=os.path.join(save_dir, 'logs')) if config.ENABLE_TENSORBOARD else None

metrics_history = []
best_recall = 0.0
lambda_transform = config.LAMBDA_TRANSFORM

# Training loop
for epoch in range(config.NUM_EPOCHS):
    print(f"\\nEpoch {epoch+1}/{config.NUM_EPOCHS}")
    train_loss = train_epoch_sketch(model_3, train_loader, optimizer, device, config.TEMPERATURE, lambda_transform)
    val_loss = validate_epoch_sketch(model_3, val_loader, device, config.TEMPERATURE)
    retrieval_metrics = evaluate_sketch_model(model_3, val_loader, device, config.K_VALUES)
    scheduler.step()

    print(f"  Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    print(f"  Val Recall@1: {retrieval_metrics['recall@1']:.2f}, mAP@1: {retrieval_metrics['mAP@1']:.2f}")

    epoch_metrics = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                     **{f'val_{k}': v for k, v in retrieval_metrics.items()}}
    metrics_history.append(epoch_metrics)

    if writer:
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        for k, v in retrieval_metrics.items():
            writer.add_scalar(f'Val/{k}', v, epoch)

    if retrieval_metrics['recall@1'] > best_recall:
        best_recall = retrieval_metrics['recall@1']
        save_checkpoint(model_3, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'best.pth')
        print(f"  New best Recall@1: {best_recall:.2f}")

    save_checkpoint(model_3, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'latest.pth')
    if (epoch + 1) % config.SAVE_EVERY_N_EPOCHS == 0:
        save_checkpoint(model_3, optimizer, scheduler, epoch, retrieval_metrics, save_dir, f'epoch_{epoch+1}.pth')

# Save metrics
metrics_df = pd.DataFrame(metrics_history)
metrics_df.to_csv(os.path.join(save_dir, 'metrics.csv'), index=False)
plot_training_curves(metrics_df, save_dir)

if writer: writer.close()

print(f"\\n{'='*80}\\n{model_name} training complete!\\nBest Val Recall@1: {best_recall:.2f}%\\n{'='*80}")

In [ ]:
cleanup_memory()

In [ ]:
# Cell 20: Validate Model 3 - ViT-PointNet (proj)
print(f"\\nFinal validation for: {model_name}")
print(f"{'='*80}")

# Load best checkpoint
load_checkpoint(model_3, None, None, os.path.join(save_dir, 'best.pth'))

# Evaluate on validation set
final_metrics = evaluate_sketch_model(model_3, val_loader, device, config.K_VALUES)

print(f"\\nFinal Validation Results:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.2f}%")

# Store results
all_training_results.append({
    'name': model_name,
    'type': 'Sketch',
    'best_val_recall@1': best_recall,
    'final_metrics': final_metrics,
    'save_dir': save_dir
})

# Cleanup
del model_3, optimizer, scheduler, train_loader, val_loader, train_dataset, val_dataset
if writer: del writer
cleanup_memory()

print(f"\\n{'='*80}\\nModel 3 complete and memory cleaned!\\n{'='*80}")

In [ ]:
# Cell 21: Train Model 4 - ViT-PointNet (no proj)
model_name = 'ViT-PointNet (no proj)'
print(f"\\n{'='*80}\\nTraining: {model_name}\\n{'='*80}")

save_dir = os.path.join(config.OUTPUT_DIR, model_name.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))
os.makedirs(save_dir, exist_ok=True)

'''# Create datasets
train_dataset = MMCADSketchDataset(train_df, config)
val_dataset = MMCADSketchDataset(val_df, config)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
                          num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                        num_workers=config.NUM_WORKERS, pin_memory=True)
'''
# Create model
model_4 = SketchPointCloudNetwork(
    pc_encoder_type='pointnet',
    latent_size=config.LATENT_SIZE,
    use_sketch_projection=False,
    use_pc_projection=True,
    freeze_vit=False,
    dgcnn_k=config.DGCNN_K
).to(device)

params = count_parameters(model_4)
print(f"Parameters: Total={params['total']:,}, Trainable={params['trainable']:,}")

# Setup training
optimizer = optim.Adam(model_4.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=config.LR_STEP_SIZE, gamma=config.LR_GAMMA)
writer = SummaryWriter(log_dir=os.path.join(save_dir, 'logs')) if config.ENABLE_TENSORBOARD else None

metrics_history = []
best_recall = 0.0
lambda_transform = config.LAMBDA_TRANSFORM

# Training loop
for epoch in range(config.NUM_EPOCHS):
    print(f"\\nEpoch {epoch+1}/{config.NUM_EPOCHS}")
    train_loss = train_epoch_sketch(model_4, train_loader, optimizer, device, config.TEMPERATURE, lambda_transform)
    val_loss = validate_epoch_sketch(model_4, val_loader, device, config.TEMPERATURE)
    retrieval_metrics = evaluate_sketch_model(model_4, val_loader, device, config.K_VALUES)
    scheduler.step()

    print(f"  Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    print(f"  Val Recall@1: {retrieval_metrics['recall@1']:.2f}, mAP@1: {retrieval_metrics['mAP@1']:.2f}")

    epoch_metrics = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                     **{f'val_{k}': v for k, v in retrieval_metrics.items()}}
    metrics_history.append(epoch_metrics)

    if writer:
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        for k, v in retrieval_metrics.items():
            writer.add_scalar(f'Val/{k}', v, epoch)

    if retrieval_metrics['recall@1'] > best_recall:
        best_recall = retrieval_metrics['recall@1']
        save_checkpoint(model_4, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'best.pth')
        print(f"  New best Recall@1: {best_recall:.2f}")

    save_checkpoint(model_4, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'latest.pth')
    if (epoch + 1) % config.SAVE_EVERY_N_EPOCHS == 0:
        save_checkpoint(model_4, optimizer, scheduler, epoch, retrieval_metrics, save_dir, f'epoch_{epoch+1}.pth')

# Save metrics
metrics_df = pd.DataFrame(metrics_history)
metrics_df.to_csv(os.path.join(save_dir, 'metrics.csv'), index=False)
plot_training_curves(metrics_df, save_dir)

if writer: writer.close()

print(f"\\n{'='*80}\\n{model_name} training complete!\\nBest Val Recall@1: {best_recall:.2f}%\\n{'='*80}")

In [ ]:
# Cell 22: Validate Model 4 - ViT-PointNet (no proj)
print(f"\\nFinal validation for: {model_name}")
print(f"{'='*80}")

# Load best checkpoint
load_checkpoint(model_4, None, None, os.path.join(save_dir, 'best.pth'))

# Evaluate on validation set
final_metrics = evaluate_sketch_model(model_4, val_loader, device, config.K_VALUES)

print(f"\\nFinal Validation Results:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.2f}%")

# Store results
all_training_results.append({
    'name': model_name,
    'type': 'Sketch',
    'best_val_recall@1': best_recall,
    'final_metrics': final_metrics,
    'save_dir': save_dir
})

# Cleanup
del model_4, optimizer, scheduler, train_loader, val_loader, train_dataset, val_dataset
if writer: del writer
cleanup_memory()

print(f"\\n{'='*80}\\nModel 4 complete and memory cleaned!\\n{'='*80}")

In [ ]:
cleanup_memory()

In [ ]:
# Cell 23: Train Model 5 - ViT-DGCNN (proj)
model_name = 'ViT-DGCNN (proj)'
print(f"\\n{'='*80}\\nTraining: {model_name}\\n{'='*80}")

save_dir = os.path.join(config.OUTPUT_DIR, model_name.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))
os.makedirs(save_dir, exist_ok=True)

'''# Create datasets
train_dataset = MMCADSketchDataset(train_df, config)
val_dataset = MMCADSketchDataset(val_df, config)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
                          num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                        num_workers=config.NUM_WORKERS, pin_memory=True)'''
train_dataset = OptimizedSketchDataset(train_df, config, cache_ply=True)
val_dataset = OptimizedSketchDataset(val_df, config, cache_ply=True)
train_loader, val_loader = create_dataloaders(train_dataset, val_dataset, config)

# Create model
model_5 = SketchPointCloudNetwork(
    pc_encoder_type='dgcnn',
    latent_size=config.LATENT_SIZE,
    use_sketch_projection=True,
    use_pc_projection=True,
    freeze_vit=False,
    dgcnn_k=config.DGCNN_K
).to(device)

params = count_parameters(model_5)
print(f"Parameters: Total={params['total']:,}, Trainable={params['trainable']:,}")

# Setup training
optimizer = optim.Adam(model_5.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=config.LR_STEP_SIZE, gamma=config.LR_GAMMA)
writer = SummaryWriter(log_dir=os.path.join(save_dir, 'logs')) if config.ENABLE_TENSORBOARD else None

metrics_history = []
best_recall = 0.0
lambda_transform = 0.0

# Training loop
for epoch in range(config.NUM_EPOCHS):
    print(f"\\nEpoch {epoch+1}/{config.NUM_EPOCHS}")
    train_loss = train_epoch_sketch(model_5, train_loader, optimizer, device, config.TEMPERATURE, lambda_transform)
    val_loss = validate_epoch_sketch(model_5, val_loader, device, config.TEMPERATURE)
    retrieval_metrics = evaluate_sketch_model(model_5, val_loader, device, config.K_VALUES)
    scheduler.step()

    print(f"  Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    print(f"  Val Recall@1: {retrieval_metrics['recall@1']:.2f}, mAP@1: {retrieval_metrics['mAP@1']:.2f}")

    epoch_metrics = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                     **{f'val_{k}': v for k, v in retrieval_metrics.items()}}
    metrics_history.append(epoch_metrics)

    if writer:
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        for k, v in retrieval_metrics.items():
            writer.add_scalar(f'Val/{k}', v, epoch)

    if retrieval_metrics['recall@1'] > best_recall:
        best_recall = retrieval_metrics['recall@1']
        save_checkpoint(model_5, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'best.pth')
        print(f"  New best Recall@1: {best_recall:.2f}")

    save_checkpoint(model_5, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'latest.pth')
    if (epoch + 1) % config.SAVE_EVERY_N_EPOCHS == 0:
        save_checkpoint(model_5, optimizer, scheduler, epoch, retrieval_metrics, save_dir, f'epoch_{epoch+1}.pth')

# Save metrics
metrics_df = pd.DataFrame(metrics_history)
metrics_df.to_csv(os.path.join(save_dir, 'metrics.csv'), index=False)
plot_training_curves(metrics_df, save_dir)

if writer: writer.close()

print(f"\\n{'='*80}\\n{model_name} training complete!\\nBest Val Recall@1: {best_recall:.2f}%\\n{'='*80}")

In [ ]:
# Cell 24: Validate Model 5 - ViT-DGCNN (proj)
print(f"\\nFinal validation for: {model_name}")
print(f"{'='*80}")

# Load best checkpoint
load_checkpoint(model_5, None, None, os.path.join(save_dir, 'best.pth'))

# Evaluate on validation set
final_metrics = evaluate_sketch_model(model_5, val_loader, device, config.K_VALUES)

print(f"\\nFinal Validation Results:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.2f}%")

# Store results
all_training_results.append({
    'name': model_name,
    'type': 'Sketch',
    'best_val_recall@1': best_recall,
    'final_metrics': final_metrics,
    'save_dir': save_dir
})

# Cleanup
del model_5, optimizer, scheduler, train_loader, val_loader, train_dataset, val_dataset
if writer: del writer
cleanup_memory()

print(f"\\n{'='*80}\\nModel 5 complete and memory cleaned!\\n{'='*80}")

In [ ]:
# Cell 25: Train Model 6 - ViT-DGCNN (no proj)
model_name = 'ViT-DGCNN (no proj)'
print(f"\\n{'='*80}\\nTraining: {model_name}\\n{'='*80}")

save_dir = os.path.join(config.OUTPUT_DIR, model_name.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))
os.makedirs(save_dir, exist_ok=True)

'''# Create datasets
train_dataset = MMCADSketchDataset(train_df, config)
val_dataset = MMCADSketchDataset(val_df, config)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
                          num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                        num_workers=config.NUM_WORKERS, pin_memory=True)'''

# Create model
model_6 = SketchPointCloudNetwork(
    pc_encoder_type='dgcnn',
    latent_size=config.LATENT_SIZE,
    use_sketch_projection=False,
    use_pc_projection=True,
    freeze_vit=False,
    dgcnn_k=config.DGCNN_K
).to(device)

params = count_parameters(model_6)
print(f"Parameters: Total={params['total']:,}, Trainable={params['trainable']:,}")

# Setup training
optimizer = optim.Adam(model_6.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=config.LR_STEP_SIZE, gamma=config.LR_GAMMA)
writer = SummaryWriter(log_dir=os.path.join(save_dir, 'logs')) if config.ENABLE_TENSORBOARD else None

metrics_history = []
best_recall = 0.0
lambda_transform = 0.0

# Training loop
for epoch in range(config.NUM_EPOCHS):
    print(f"\\nEpoch {epoch+1}/{config.NUM_EPOCHS}")
    train_loss = train_epoch_sketch(model_6, train_loader, optimizer, device, config.TEMPERATURE, lambda_transform)
    val_loss = validate_epoch_sketch(model_6, val_loader, device, config.TEMPERATURE)
    retrieval_metrics = evaluate_sketch_model(model_6, val_loader, device, config.K_VALUES)
    scheduler.step()

    print(f"  Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    print(f"  Val Recall@1: {retrieval_metrics['recall@1']:.2f}, mAP@1: {retrieval_metrics['mAP@1']:.2f}")

    epoch_metrics = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                     **{f'val_{k}': v for k, v in retrieval_metrics.items()}}
    metrics_history.append(epoch_metrics)

    if writer:
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        for k, v in retrieval_metrics.items():
            writer.add_scalar(f'Val/{k}', v, epoch)

    if retrieval_metrics['recall@1'] > best_recall:
        best_recall = retrieval_metrics['recall@1']
        save_checkpoint(model_6, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'best.pth')
        print(f"  New best Recall@1: {best_recall:.2f}")

    save_checkpoint(model_6, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'latest.pth')
    if (epoch + 1) % config.SAVE_EVERY_N_EPOCHS == 0:
        save_checkpoint(model_6, optimizer, scheduler, epoch, retrieval_metrics, save_dir, f'epoch_{epoch+1}.pth')

# Save metrics
metrics_df = pd.DataFrame(metrics_history)
metrics_df.to_csv(os.path.join(save_dir, 'metrics.csv'), index=False)
plot_training_curves(metrics_df, save_dir)

if writer: writer.close()

print(f"\\n{'='*80}\\n{model_name} training complete!\\nBest Val Recall@1: {best_recall:.2f}%\\n{'='*80}")

In [ ]:
# Cell 26: Validate Model 6 - ViT-DGCNN (no proj)
print(f"\\nFinal validation for: {model_name}")
print(f"{'='*80}")

# Load best checkpoint
load_checkpoint(model_6, None, None, os.path.join(save_dir, 'best.pth'))

# Evaluate on validation set
final_metrics = evaluate_sketch_model(model_6, val_loader, device, config.K_VALUES)

print(f"\\nFinal Validation Results:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.2f}%")

# Store results
all_training_results.append({
    'name': model_name,
    'type': 'Sketch',
    'best_val_recall@1': best_recall,
    'final_metrics': final_metrics,
    'save_dir': save_dir
})

# Cleanup
del model_6, optimizer, scheduler, train_loader, val_loader, train_dataset, val_dataset
if writer: del writer
cleanup_memory()

print(f"\\n{'='*80}\\nModel 6 complete and memory cleaned!\\n{'='*80}")

In [ ]:
cleanup_memory()

---\n# Text-to-PointCloud Models\n## 6 models using BERT/CLIP for text encoding

In [ ]:
# Cell 27: Train Model 7 - BERT-PointNetMini
model_name = 'BERT-PointNetMini'
print(f"\\n{'='*80}\\nTraining: {model_name}\\n{'='*80}")

save_dir = os.path.join(config.OUTPUT_DIR, model_name.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))
os.makedirs(save_dir, exist_ok=True)

# Create datasets
tokenizer = BertTokenizer.from_pretrained(config.BERT_MODEL)
train_dataset = OptimizedTextDataset(train_df, config, tokenizer, cache_ply=True)
val_dataset = OptimizedTextDataset(val_df, config, tokenizer, cache_ply=True)
train_loader, val_loader = create_dataloaders(train_dataset, val_dataset, config)

# Create model
model_7 = TextPointCloudNetwork(
    text_encoder_type='bert',
    pc_encoder_type='pointcloud',
    latent_size=config.LATENT_SIZE,
    freeze_text=False,
    dgcnn_k=config.DGCNN_K
).to(device)

params = count_parameters(model_7)
print(f"Parameters: Total={params['total']:,}, Trainable={params['trainable']:,}")

# Setup training
optimizer = optim.Adam(model_7.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=config.LR_STEP_SIZE, gamma=config.LR_GAMMA)
writer = SummaryWriter(log_dir=os.path.join(save_dir, 'logs')) if config.ENABLE_TENSORBOARD else None

metrics_history = []
best_recall = 0.0
lambda_transform = 0.0

# Training loop
for epoch in range(config.NUM_EPOCHS):
    print(f"\\nEpoch {epoch+1}/{config.NUM_EPOCHS}")
    train_loss = train_epoch_text(model_7, train_loader, optimizer, device, config.TEMPERATURE, lambda_transform)
    val_loss = validate_epoch_text(model_7, val_loader, device, config.TEMPERATURE)
    retrieval_metrics = evaluate_text_model(model_7, val_loader, device, config.K_VALUES)
    scheduler.step()

    print(f"  Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    print(f"  Val Recall@1: {retrieval_metrics['recall@1']:.2f}, mAP@1: {retrieval_metrics['mAP@1']:.2f}")

    epoch_metrics = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                     **{f'val_{k}': v for k, v in retrieval_metrics.items()}}
    metrics_history.append(epoch_metrics)

    if writer:
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        for k, v in retrieval_metrics.items():
            writer.add_scalar(f'Val/{k}', v, epoch)

    if retrieval_metrics['recall@1'] > best_recall:
        best_recall = retrieval_metrics['recall@1']
        save_checkpoint(model_7, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'best.pth')
        print(f"  New best Recall@1: {best_recall:.2f}")

    save_checkpoint(model_7, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'latest.pth')
    if (epoch + 1) % config.SAVE_EVERY_N_EPOCHS == 0:
        save_checkpoint(model_7, optimizer, scheduler, epoch, retrieval_metrics, save_dir, f'epoch_{epoch+1}.pth')

# Save metrics
metrics_df = pd.DataFrame(metrics_history)
metrics_df.to_csv(os.path.join(save_dir, 'metrics.csv'), index=False)
plot_training_curves(metrics_df, save_dir)

if writer: writer.close()

print(f"\\n{'='*80}\\n{model_name} training complete!\\nBest Val Recall@1: {best_recall:.2f}%\\n{'='*80}")

In [ ]:
# Cell 28: Validate Model 7 - BERT-PointNetMini
print(f"\\nFinal validation for: {model_name}")
print(f"{'='*80}")

# Load best checkpoint
load_checkpoint(model_7, None, None, os.path.join(save_dir, 'best.pth'))

# Evaluate on validation set
final_metrics = evaluate_text_model(model_7, val_loader, device, config.K_VALUES)

print(f"\\nFinal Validation Results:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.2f}%")

# Store results
all_training_results.append({
    'name': model_name,
    'type': 'Text',
    'best_val_recall@1': best_recall,
    'final_metrics': final_metrics,
    'save_dir': save_dir
})

# Cleanup
del model_7, optimizer, scheduler, train_loader, val_loader, train_dataset, val_dataset, tokenizer
if writer: del writer
cleanup_memory()

print(f"\\n{'='*80}\\nModel 7 complete and memory cleaned!\\n{'='*80}")

In [ ]:
cleanup_memory()

In [ ]:
# Cell 29: Train Model 8 - BERT-PointNet
model_name = 'BERT-PointNet'
print(f"\\n{'='*80}\\nTraining: {model_name}\\n{'='*80}")

save_dir = os.path.join(config.OUTPUT_DIR, model_name.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))
os.makedirs(save_dir, exist_ok=True)

'''# Create datasets
tokenizer = BertTokenizer.from_pretrained(config.BERT_MODEL)
train_dataset = MMCADTextDataset(train_df, config, tokenizer)
val_dataset = MMCADTextDataset(val_df, config, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
                          num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                        num_workers=config.NUM_WORKERS, pin_memory=True)'''

# Create model
model_8 = TextPointCloudNetwork(
    text_encoder_type='bert',
    pc_encoder_type='pointnet',
    latent_size=config.LATENT_SIZE,
    freeze_text=False,
    dgcnn_k=config.DGCNN_K
).to(device)

params = count_parameters(model_8)
print(f"Parameters: Total={params['total']:,}, Trainable={params['trainable']:,}")

# Setup training
optimizer = optim.Adam(model_8.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=config.LR_STEP_SIZE, gamma=config.LR_GAMMA)
writer = SummaryWriter(log_dir=os.path.join(save_dir, 'logs')) if config.ENABLE_TENSORBOARD else None

metrics_history = []
best_recall = 0.0
lambda_transform = config.LAMBDA_TRANSFORM

# Training loop
for epoch in range(config.NUM_EPOCHS):
    print(f"\\nEpoch {epoch+1}/{config.NUM_EPOCHS}")
    train_loss = train_epoch_text(model_8, train_loader, optimizer, device, config.TEMPERATURE, lambda_transform)
    val_loss = validate_epoch_text(model_8, val_loader, device, config.TEMPERATURE)
    retrieval_metrics = evaluate_text_model(model_8, val_loader, device, config.K_VALUES)
    scheduler.step()

    print(f"  Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    print(f"  Val Recall@1: {retrieval_metrics['recall@1']:.2f}, mAP@1: {retrieval_metrics['mAP@1']:.2f}")

    epoch_metrics = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                     **{f'val_{k}': v for k, v in retrieval_metrics.items()}}
    metrics_history.append(epoch_metrics)

    if writer:
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        for k, v in retrieval_metrics.items():
            writer.add_scalar(f'Val/{k}', v, epoch)

    if retrieval_metrics['recall@1'] > best_recall:
        best_recall = retrieval_metrics['recall@1']
        save_checkpoint(model_8, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'best.pth')
        print(f"  New best Recall@1: {best_recall:.2f}")

    save_checkpoint(model_8, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'latest.pth')
    if (epoch + 1) % config.SAVE_EVERY_N_EPOCHS == 0:
        save_checkpoint(model_8, optimizer, scheduler, epoch, retrieval_metrics, save_dir, f'epoch_{epoch+1}.pth')

# Save metrics
metrics_df = pd.DataFrame(metrics_history)
metrics_df.to_csv(os.path.join(save_dir, 'metrics.csv'), index=False)
plot_training_curves(metrics_df, save_dir)

if writer: writer.close()

print(f"\\n{'='*80}\\n{model_name} training complete!\\nBest Val Recall@1: {best_recall:.2f}%\\n{'='*80}")

In [ ]:
# Cell 30: Validate Model 8 - BERT-PointNet
print(f"\\nFinal validation for: {model_name}")
print(f"{'='*80}")

# Load best checkpoint
load_checkpoint(model_8, None, None, os.path.join(save_dir, 'best.pth'))

# Evaluate on validation set
final_metrics = evaluate_text_model(model_8, val_loader, device, config.K_VALUES)

print(f"\\nFinal Validation Results:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.2f}%")

# Store results
all_training_results.append({
    'name': model_name,
    'type': 'Text',
    'best_val_recall@1': best_recall,
    'final_metrics': final_metrics,
    'save_dir': save_dir
})

# Cleanup
del model_8, optimizer, scheduler, train_loader, val_loader, train_dataset, val_dataset, tokenizer
if writer: del writer
cleanup_memory()

print(f"\\n{'='*80}\\nModel 8 complete and memory cleaned!\\n{'='*80}")

In [ ]:
cleanup_memory()

In [ ]:
# Cell 31: Train Model 9 - BERT-DGCNN
model_name = 'BERT-DGCNN'
print(f"\\n{'='*80}\\nTraining: {model_name}\\n{'='*80}")

save_dir = os.path.join(config.OUTPUT_DIR, model_name.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))
os.makedirs(save_dir, exist_ok=True)

''' Create datasets
tokenizer = BertTokenizer.from_pretrained(config.BERT_MODEL)
train_dataset = MMCADTextDataset(train_df, config, tokenizer)
val_dataset = MMCADTextDataset(val_df, config, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
                          num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                        num_workers=config.NUM_WORKERS, pin_memory=True)
'''
# Create model
model_9 = TextPointCloudNetwork(
    text_encoder_type='bert',
    pc_encoder_type='dgcnn',
    latent_size=config.LATENT_SIZE,
    freeze_text=False,
    dgcnn_k=config.DGCNN_K
).to(device)

params = count_parameters(model_9)
print(f"Parameters: Total={params['total']:,}, Trainable={params['trainable']:,}")

# Setup training
optimizer = optim.Adam(model_9.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=config.LR_STEP_SIZE, gamma=config.LR_GAMMA)
writer = SummaryWriter(log_dir=os.path.join(save_dir, 'logs')) if config.ENABLE_TENSORBOARD else None

metrics_history = []
best_recall = 0.0
lambda_transform = 0.0

# Training loop
for epoch in range(config.NUM_EPOCHS):
    print(f"\\nEpoch {epoch+1}/{config.NUM_EPOCHS}")
    train_loss = train_epoch_text(model_9, train_loader, optimizer, device, config.TEMPERATURE, lambda_transform)
    val_loss = validate_epoch_text(model_9, val_loader, device, config.TEMPERATURE)
    retrieval_metrics = evaluate_text_model(model_9, val_loader, device, config.K_VALUES)
    scheduler.step()

    print(f"  Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    print(f"  Val Recall@1: {retrieval_metrics['recall@1']:.2f}, mAP@1: {retrieval_metrics['mAP@1']:.2f}")

    epoch_metrics = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                     **{f'val_{k}': v for k, v in retrieval_metrics.items()}}
    metrics_history.append(epoch_metrics)

    if writer:
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        for k, v in retrieval_metrics.items():
            writer.add_scalar(f'Val/{k}', v, epoch)

    if retrieval_metrics['recall@1'] > best_recall:
        best_recall = retrieval_metrics['recall@1']
        save_checkpoint(model_9, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'best.pth')
        print(f"  New best Recall@1: {best_recall:.2f}")

    save_checkpoint(model_9, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'latest.pth')
    if (epoch + 1) % config.SAVE_EVERY_N_EPOCHS == 0:
        save_checkpoint(model_9, optimizer, scheduler, epoch, retrieval_metrics, save_dir, f'epoch_{epoch+1}.pth')

# Save metrics
metrics_df = pd.DataFrame(metrics_history)
metrics_df.to_csv(os.path.join(save_dir, 'metrics.csv'), index=False)
plot_training_curves(metrics_df, save_dir)

if writer: writer.close()

print(f"\\n{'='*80}\\n{model_name} training complete!\\nBest Val Recall@1: {best_recall:.2f}%\\n{'='*80}")

In [ ]:
# Cell 32: Validate Model 9 - BERT-DGCNN
print(f"\\nFinal validation for: {model_name}")
print(f"{'='*80}")

# Load best checkpoint
load_checkpoint(model_9, None, None, os.path.join(save_dir, 'best.pth'))

# Evaluate on validation set
final_metrics = evaluate_text_model(model_9, val_loader, device, config.K_VALUES)

print(f"\\nFinal Validation Results:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.2f}%")

# Store results
all_training_results.append({
    'name': model_name,
    'type': 'Text',
    'best_val_recall@1': best_recall,
    'final_metrics': final_metrics,
    'save_dir': save_dir
})

# Cleanup
del model_9, optimizer, scheduler, train_loader, val_loader, train_dataset, val_dataset, tokenizer
if writer: del writer
cleanup_memory()

print(f"\\n{'='*80}\\nModel 9 complete and memory cleaned!\\n{'='*80}")

In [ ]:
cleanup_memory()

In [ ]:
# Cell 33: Train Model 10 - CLIP-PointNetMini
model_name = 'CLIP-PointNetMini'
print(f"\\n{'='*80}\\nTraining: {model_name}\\n{'='*80}")

save_dir = os.path.join(config.OUTPUT_DIR, model_name.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))
os.makedirs(save_dir, exist_ok=True)

# Create datasets
tokenizer = CLIPTokenizer.from_pretrained(config.CLIP_MODEL)
train_dataset = MMCADTextDataset(train_df, config, tokenizer)
val_dataset = MMCADTextDataset(val_df, config, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
                          num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                        num_workers=config.NUM_WORKERS, pin_memory=True)

# Create model
model_10 = TextPointCloudNetwork(
    text_encoder_type='clip',
    pc_encoder_type='pointcloud',
    latent_size=config.LATENT_SIZE_CLIP,
    freeze_text=False,
    dgcnn_k=config.DGCNN_K
).to(device)

params = count_parameters(model_10)
print(f"Parameters: Total={params['total']:,}, Trainable={params['trainable']:,}")

# Setup training
optimizer = optim.Adam(model_10.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=config.LR_STEP_SIZE, gamma=config.LR_GAMMA)
writer = SummaryWriter(log_dir=os.path.join(save_dir, 'logs')) if config.ENABLE_TENSORBOARD else None

metrics_history = []
best_recall = 0.0
lambda_transform = 0.0

# Training loop
for epoch in range(config.NUM_EPOCHS):
    print(f"\\nEpoch {epoch+1}/{config.NUM_EPOCHS}")
    train_loss = train_epoch_text(model_10, train_loader, optimizer, device, config.TEMPERATURE, lambda_transform)
    val_loss = validate_epoch_text(model_10, val_loader, device, config.TEMPERATURE)
    retrieval_metrics = evaluate_text_model(model_10, val_loader, device, config.K_VALUES)
    scheduler.step()

    print(f"  Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    print(f"  Val Recall@1: {retrieval_metrics['recall@1']:.2f}, mAP@1: {retrieval_metrics['mAP@1']:.2f}")

    epoch_metrics = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                     **{f'val_{k}': v for k, v in retrieval_metrics.items()}}
    metrics_history.append(epoch_metrics)

    if writer:
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        for k, v in retrieval_metrics.items():
            writer.add_scalar(f'Val/{k}', v, epoch)

    if retrieval_metrics['recall@1'] > best_recall:
        best_recall = retrieval_metrics['recall@1']
        save_checkpoint(model_10, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'best.pth')
        print(f"  New best Recall@1: {best_recall:.2f}")

    save_checkpoint(model_10, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'latest.pth')
    if (epoch + 1) % config.SAVE_EVERY_N_EPOCHS == 0:
        save_checkpoint(model_10, optimizer, scheduler, epoch, retrieval_metrics, save_dir, f'epoch_{epoch+1}.pth')

# Save metrics
metrics_df = pd.DataFrame(metrics_history)
metrics_df.to_csv(os.path.join(save_dir, 'metrics.csv'), index=False)
plot_training_curves(metrics_df, save_dir)

if writer: writer.close()

print(f"\\n{'='*80}\\n{model_name} training complete!\\nBest Val Recall@1: {best_recall:.2f}%\\n{'='*80}")

In [ ]:
# Cell 34: Validate Model 10 - CLIP-PointNetMini
print(f"\\nFinal validation for: {model_name}")
print(f"{'='*80}")

# Load best checkpoint
load_checkpoint(model_10, None, None, os.path.join(save_dir, 'best.pth'))

# Evaluate on validation set
final_metrics = evaluate_text_model(model_10, val_loader, device, config.K_VALUES)

print(f"\\nFinal Validation Results:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.2f}%")

# Store results
all_training_results.append({
    'name': model_name,
    'type': 'Text',
    'best_val_recall@1': best_recall,
    'final_metrics': final_metrics,
    'save_dir': save_dir
})

# Cleanup
del model_10, optimizer, scheduler, train_loader, val_loader, train_dataset, val_dataset, tokenizer
if writer: del writer
cleanup_memory()

print(f"\\n{'='*80}\\nModel 10 complete and memory cleaned!\\n{'='*80}")

In [ ]:
# Cell 35: Train Model 11 - CLIP-PointNet
model_name = 'CLIP-PointNet'
print(f"\\n{'='*80}\\nTraining: {model_name}\\n{'='*80}")

save_dir = os.path.join(config.OUTPUT_DIR, model_name.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))
os.makedirs(save_dir, exist_ok=True)

# Create datasets
tokenizer = CLIPTokenizer.from_pretrained(config.CLIP_MODEL)
train_dataset = MMCADTextDataset(train_df, config, tokenizer)
val_dataset = MMCADTextDataset(val_df, config, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
                          num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                        num_workers=config.NUM_WORKERS, pin_memory=True)

# Create model
model_11 = TextPointCloudNetwork(
    text_encoder_type='clip',
    pc_encoder_type='pointnet',
    latent_size=config.LATENT_SIZE_CLIP,
    freeze_text=False,
    dgcnn_k=config.DGCNN_K
).to(device)

params = count_parameters(model_11)
print(f"Parameters: Total={params['total']:,}, Trainable={params['trainable']:,}")

# Setup training
optimizer = optim.Adam(model_11.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=config.LR_STEP_SIZE, gamma=config.LR_GAMMA)
writer = SummaryWriter(log_dir=os.path.join(save_dir, 'logs')) if config.ENABLE_TENSORBOARD else None

metrics_history = []
best_recall = 0.0
lambda_transform = config.LAMBDA_TRANSFORM

# Training loop
for epoch in range(config.NUM_EPOCHS):
    print(f"\\nEpoch {epoch+1}/{config.NUM_EPOCHS}")
    train_loss = train_epoch_text(model_11, train_loader, optimizer, device, config.TEMPERATURE, lambda_transform)
    val_loss = validate_epoch_text(model_11, val_loader, device, config.TEMPERATURE)
    retrieval_metrics = evaluate_text_model(model_11, val_loader, device, config.K_VALUES)
    scheduler.step()

    print(f"  Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    print(f"  Val Recall@1: {retrieval_metrics['recall@1']:.2f}, mAP@1: {retrieval_metrics['mAP@1']:.2f}")

    epoch_metrics = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                     **{f'val_{k}': v for k, v in retrieval_metrics.items()}}
    metrics_history.append(epoch_metrics)

    if writer:
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        for k, v in retrieval_metrics.items():
            writer.add_scalar(f'Val/{k}', v, epoch)

    if retrieval_metrics['recall@1'] > best_recall:
        best_recall = retrieval_metrics['recall@1']
        save_checkpoint(model_11, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'best.pth')
        print(f"  New best Recall@1: {best_recall:.2f}")

    save_checkpoint(model_11, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'latest.pth')
    if (epoch + 1) % config.SAVE_EVERY_N_EPOCHS == 0:
        save_checkpoint(model_11, optimizer, scheduler, epoch, retrieval_metrics, save_dir, f'epoch_{epoch+1}.pth')

# Save metrics
metrics_df = pd.DataFrame(metrics_history)
metrics_df.to_csv(os.path.join(save_dir, 'metrics.csv'), index=False)
plot_training_curves(metrics_df, save_dir)

if writer: writer.close()

print(f"\\n{'='*80}\\n{model_name} training complete!\\nBest Val Recall@1: {best_recall:.2f}%\\n{'='*80}")

In [ ]:
# Cell 36: Validate Model 11 - CLIP-PointNet
print(f"\\nFinal validation for: {model_name}")
print(f"{'='*80}")

# Load best checkpoint
load_checkpoint(model_11, None, None, os.path.join(save_dir, 'best.pth'))

# Evaluate on validation set
final_metrics = evaluate_text_model(model_11, val_loader, device, config.K_VALUES)

print(f"\\nFinal Validation Results:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.2f}%")

# Store results
all_training_results.append({
    'name': model_name,
    'type': 'Text',
    'best_val_recall@1': best_recall,
    'final_metrics': final_metrics,
    'save_dir': save_dir
})

# Cleanup
del model_11, optimizer, scheduler, train_loader, val_loader, train_dataset, val_dataset, tokenizer
if writer: del writer
cleanup_memory()

print(f"\\n{'='*80}\\nModel 11 complete and memory cleaned!\\n{'='*80}")

In [ ]:
# Cell 37: Train Model 12 - CLIP-DGCNN
model_name = 'CLIP-DGCNN'
print(f"\\n{'='*80}\\nTraining: {model_name}\\n{'='*80}")

save_dir = os.path.join(config.OUTPUT_DIR, model_name.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))
os.makedirs(save_dir, exist_ok=True)

# Create datasets
tokenizer = CLIPTokenizer.from_pretrained(config.CLIP_MODEL)
train_dataset = MMCADTextDataset(train_df, config, tokenizer)
val_dataset = MMCADTextDataset(val_df, config, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
                          num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                        num_workers=config.NUM_WORKERS, pin_memory=True)

# Create model
model_12 = TextPointCloudNetwork(
    text_encoder_type='clip',
    pc_encoder_type='dgcnn',
    latent_size=config.LATENT_SIZE_CLIP,
    freeze_text=False,
    dgcnn_k=config.DGCNN_K
).to(device)

params = count_parameters(model_12)
print(f"Parameters: Total={params['total']:,}, Trainable={params['trainable']:,}")

# Setup training
optimizer = optim.Adam(model_12.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=config.LR_STEP_SIZE, gamma=config.LR_GAMMA)
writer = SummaryWriter(log_dir=os.path.join(save_dir, 'logs')) if config.ENABLE_TENSORBOARD else None

metrics_history = []
best_recall = 0.0
lambda_transform = 0.0

# Training loop
for epoch in range(config.NUM_EPOCHS):
    print(f"\\nEpoch {epoch+1}/{config.NUM_EPOCHS}")
    train_loss = train_epoch_text(model_12, train_loader, optimizer, device, config.TEMPERATURE, lambda_transform)
    val_loss = validate_epoch_text(model_12, val_loader, device, config.TEMPERATURE)
    retrieval_metrics = evaluate_text_model(model_12, val_loader, device, config.K_VALUES)
    scheduler.step()

    print(f"  Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    print(f"  Val Recall@1: {retrieval_metrics['recall@1']:.2f}, mAP@1: {retrieval_metrics['mAP@1']:.2f}")

    epoch_metrics = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                     **{f'val_{k}': v for k, v in retrieval_metrics.items()}}
    metrics_history.append(epoch_metrics)

    if writer:
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        for k, v in retrieval_metrics.items():
            writer.add_scalar(f'Val/{k}', v, epoch)

    if retrieval_metrics['recall@1'] > best_recall:
        best_recall = retrieval_metrics['recall@1']
        save_checkpoint(model_12, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'best.pth')
        print(f"  New best Recall@1: {best_recall:.2f}")

    save_checkpoint(model_12, optimizer, scheduler, epoch, retrieval_metrics, save_dir, 'latest.pth')
    if (epoch + 1) % config.SAVE_EVERY_N_EPOCHS == 0:
        save_checkpoint(model_12, optimizer, scheduler, epoch, retrieval_metrics, save_dir, f'epoch_{epoch+1}.pth')

# Save metrics
metrics_df = pd.DataFrame(metrics_history)
metrics_df.to_csv(os.path.join(save_dir, 'metrics.csv'), index=False)
plot_training_curves(metrics_df, save_dir)

if writer: writer.close()

print(f"\\n{'='*80}\\n{model_name} training complete!\\nBest Val Recall@1: {best_recall:.2f}%\\n{'='*80}")

In [ ]:
# Cell 38: Validate Model 12 - CLIP-DGCNN
print(f"\\nFinal validation for: {model_name}")
print(f"{'='*80}")

# Load best checkpoint
load_checkpoint(model_12, None, None, os.path.join(save_dir, 'best.pth'))

# Evaluate on validation set
final_metrics = evaluate_text_model(model_12, val_loader, device, config.K_VALUES)

print(f"\\nFinal Validation Results:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.2f}%")

# Store results
all_training_results.append({
    'name': model_name,
    'type': 'Text',
    'best_val_recall@1': best_recall,
    'final_metrics': final_metrics,
    'save_dir': save_dir
})

# Cleanup
del model_12, optimizer, scheduler, train_loader, val_loader, train_dataset, val_dataset, tokenizer
if writer: del writer
cleanup_memory()

print(f"\\n{'='*80}\\nModel 12 complete and memory cleaned!\\n{'='*80}")

---\n# Training Summary\nDisplay results from all trained models

In [ ]:
# Cell: Training Summary
if all_training_results:
    print("\\n" + "="*80)
    print("TRAINING SUMMARY - ALL MODELS")
    print("="*80)
    summary_df = pd.DataFrame([{
        'Model': r['name'],
        'Type': r['type'],
        'Best Val R@1': r['best_val_recall@1'],
        **{f'{k}': v for k, v in r['final_metrics'].items()}
    } for r in all_training_results])
    print(summary_df.to_string(index=False))
    summary_df.to_csv(os.path.join(config.OUTPUT_DIR, 'training_summary.csv'), index=False)
    print(f"\\nTraining summary saved to {os.path.join(config.OUTPUT_DIR, 'training_summary.csv')}")
else:
    print("No models have been trained yet.")